# Advanced Planning Evaluation ? notebook solo grafici

Questo notebook ? una versione statica e consultabile di `advanced_planning_evaluation.ipynb`: non contiene celle di codice e non ricalcola nulla. Le figure sono caricate direttamente dalla cartella `results/` usando path relativi, quindi il notebook resta leggero e mostra sempre i PNG presenti nel repository.

Creato: 2026-07-01 10:14

Sorgenti usate:
- `../../results/nvidia_complete_api_test_RESULTS/` con metadati da `../../results/nvidia_complete_api_test_RESULTS.json`.
- `../../results/LLM_classification_test/` con metadati da `../../results/LLM_classification_test.json`.
- `../../results/prova_merge/`, disponibile come appendice figure-only perch? non ha un JSON separato nella cartella `results/`.


## Copertura dei risultati

**Run NVIDIA completo**

- Run id: `nvidia_parallel_complete_api_test`.
- Modelli: `nvidia_gpt_oss_120b, nvidia_kimi_k2_6, nvidia_llama_3_3_70b_instruct, nvidia_nemotron_3_nano_30b_a3b, nvidia_phi_4_mini_instruct, nvidia_qwen_3_5_122b_a10b`.
- Figure trovate: `27` PNG.
- Nota: ? il set pi? completo e quello da usare come lettura principale.

**LLM classification test**

- Run id: `2026-06-05_21-06-20`.
- Modelli: `hf_gemma_4_31b_it, hf_nemotron_3_nano_30b_a3b, hf_phi_4, hf_qwen_3_6_27b`.
- Figure trovate: `6` PNG.
- Nota: ? una vista compatta con 6 figure, utile come confronto separato.

**prova_merge**

- Figure trovate: `27` PNG.
- Problema interpretativo: nella cartella `results/` non c?? un JSON `prova_merge.json`, quindi non ? possibile verificare da questo notebook quali run siano stati fusi. Le immagini sono incluse, ma vanno lette come esportazione distinta.


## Legenda delle metriche

Quasi tutte le metriche sono normalizzate in `[0, 1]`. Le eccezioni principali sono `Temporal_Distance`, lunghezza media, conteggi e varianze. Le metriche non disponibili possono apparire come celle vuote, trattini o pannelli mancanti: non significano automaticamente performance zero.

| Metrica | Cosa misura | Direzione | Da leggere insieme a |
| --- | --- | --- | --- |
| SR / Success Rate | quota di problemi risolti dopo tutto il budget di tentativi | alto = meglio | FASR, Retry Gap, IWSR |
| FASR | successo al primo tentativo; stima la capacit? zero-shot | alto = meglio | SR, Retry Gap |
| IWSR | successo pesato per numero di iterazioni; premia correzioni rapide | alto = meglio | FASR, SR, P(Valid|k) |
| Retry Gap | SR - FASR; quanto il risultato dipende dai retry | basso = meglio | IWSR, P(Valid|k) |
| Exec | frazione del piano eseguibile prima del primo fallimento | alto = meglio | PAS, lunghezza piano |
| Halluc | azioni non presenti nel dominio PDDL | basso = meglio | IHR, object hallucination |
| IHR | 1 - Halluc; gate di grounding del vocabolario | alto = meglio | Exec, PAS, CoT |
| PAS | quota di errori interpretabili come sequencing invece che fabricazione di stato | alto = meglio | Exec, failure breakdown |
| CoT Alignment | quanto la reasoning trace parla delle stesse azioni/oggetti del piano | alto = meglio | SR, validit? piano |
| Temporal Distance | distanza temporale media degli errori di sequencing | basso = meglio | PAS, failure type |
| PS | score composito: FASR, IWSR, Exec, IHR, PAS, opzionale CoT | alto = meglio | tutte le componenti |


## Ordine consigliato di lettura

1. **Grounding del vocabolario:** controlla `Halluc` o `IHR`. Se il modello inventa azioni o oggetti, le metriche successive diventano fragili.
2. **Outcome:** guarda `SR`, ma non fermarti l?: SR pu? essere gonfiato dai retry.
3. **Autenticit? zero-shot:** confronta `SR`, `FASR` e `Retry Gap`.
4. **Efficienza dei retry:** usa `IWSR` e `P(Valid | k)`. Correzione rapida e re-sampling casuale possono avere SR simile ma profili diversi.
5. **Anatomia del fallimento:** leggi `Exec`, `PAS`, `failure_type_breakdown` e `Temporal_Distance` insieme.
6. **Reasoning trace:** interpreta `CoT Alignment` solo insieme a validit? del piano e grounding.
7. **Generalizzazione:** usa ranking, varianza di rank e correlazioni tra domini per distinguere modelli generalisti da specialisti.
8. **Sintesi:** usa `PS` e radar come riassunto, poi torna alle metriche componenti per spiegare il perch?.

Regola pratica: nessun grafico singolo basta per sostenere una conclusione forte. Le coppie minime sono `SR/FASR`, `FASR/IWSR`, `Halluc/Exec`, `Exec/PAS`, `CoT/SR`, `PS/componenti`.


## Tabella sintetica ? NVIDIA complete API test

Valori arrotondati a tre decimali. Questa tabella serve come riferimento numerico rapido per le figure principali.

| Model | N | SR | FASR | IWSR | Retry Gap | Exec | Halluc | IHR | PAS | CoT | PS |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| nvidia_gpt_oss_120b | 216 | 0.329 | 0.278 | 0.291 | 0.051 | 0.720 | 0.000 | 1.000 | 0.076 | 0.453 | 0.482 |
| nvidia_kimi_k2_6 | 216 | 0.074 | 0.069 | 0.071 | 0.005 | 0.925 | 0.000 | 1.000 | 0.000 | 0.858 | 0.438 |
| nvidia_llama_3_3_70b_instruct | 216 | 0.019 | 0.019 | 0.019 | 0.000 | 0.561 | 0.000 | 1.000 | 0.057 | ? | 0.329 |
| nvidia_nemotron_3_nano_30b_a3b | 216 | 0.162 | 0.116 | 0.132 | 0.046 | 0.538 | 0.000 | 1.000 | 0.147 | 0.401 | 0.386 |
| nvidia_phi_4_mini_instruct | 216 | 0.000 | 0.000 | 0.000 | 0.000 | 0.201 | 0.000 | 1.000 | 0.000 | ? | 0.240 |
| nvidia_qwen_3_5_122b_a10b | 216 | 0.088 | 0.079 | 0.083 | 0.009 | 1.000 | 0.000 | 1.000 | 0.500 | 0.813 | 0.526 |


## Outcome e retry

Figure da `../../results/nvidia_complete_api_test_RESULTS/`.


### Outcome per dominio

![Outcome per dominio](../../results/nvidia_complete_api_test_RESULTS/success_rate_heatmap.png)

**Come leggerlo:** Ogni cella mostra la quota di problemi validi per una coppia modello-dominio. Serve per capire dove un modello funziona, ma non dice se il successo arriva al primo tentativo o dopo molti retry.

**Interconnessione:** Success rate heatmap by model and domain.


### Successo totale contro primo tentativo

![Successo totale contro primo tentativo](../../results/nvidia_complete_api_test_RESULTS/sr_vs_fasr.png)

**Come leggerlo:** La distanza tra SR e FASR segnala dipendenza dal meccanismo iterativo. Se le barre sono vicine, il modello risolve soprattutto subito; se SR ? molto pi? alto, il risultato finale ? scaffolded dai retry.

**Interconnessione:** Overall success rate compared with first-attempt success rate.


### SR, FASR e IWSR insieme

![SR, FASR e IWSR insieme](../../results/nvidia_complete_api_test_RESULTS/sr_fasr_iwsr_by_model.png)

**Come leggerlo:** Usa SR come tetto finale, FASR come baseline zero-shot e IWSR come efficienza dei retry. IWSR vicino a SR indica correzioni rapide; IWSR vicino a FASR indica che i retry aggiungono poco o arrivano tardi.

**Interconnessione:** SR, FASR, and IWSR comparison per model.


### FASR per dominio

![FASR per dominio](../../results/nvidia_complete_api_test_RESULTS/fasr_by_model_domain.png)

**Come leggerlo:** Mostra la robustezza al primo tentativo. ? la vista pi? severa perch? non pu? essere gonfiata dal numero di iterazioni.

**Interconnessione:** First-attempt success rate by model and domain.


### FASR per difficolt?

![FASR per difficolt?](../../results/nvidia_complete_api_test_RESULTS/fasr_by_difficulty.png)

**Come leggerlo:** Controlla se la capacit? zero-shot degrada da easy a hard. Un crollo netto indica sensibilit? alla complessit?; una linea stabile indica maggiore generalizzazione.

**Interconnessione:** First-attempt success rate by difficulty tier.


### Efficienza delle correzioni

![Efficienza delle correzioni](../../results/nvidia_complete_api_test_RESULTS/iwsr_by_model_domain.png)

**Come leggerlo:** Confronta quanto i retry sono utili nei diversi domini. Va letto con FASR: alto IWSR con basso FASR pu? indicare buon correttore, non necessariamente buon planner zero-shot.

**Interconnessione:** Iteration-weighted success rate by model and domain.


### Dipendenza dai retry

![Dipendenza dai retry](../../results/nvidia_complete_api_test_RESULTS/retry_gap_by_model.png)

**Come leggerlo:** Valori alti significano che molti successi spariscono togliendo il meccanismo iterativo. Non interpretare Retry Gap basso come qualit? se anche SR ? vicino a zero.

**Interconnessione:** SR minus FASR; higher values mean stronger retry dependence.


### Profilo zero-shot / retry

![Profilo zero-shot / retry](../../results/nvidia_complete_api_test_RESULTS/fasr_iwsr_scatter.png)

**Come leggerlo:** Asse X = FASR, asse Y = IWSR, dimensione/colore = SR. Punti in alto a destra sono migliori; punti molto sopra la diagonale dipendono dai retry.

**Interconnessione:** FASR versus IWSR with success rate encoded by dot size and color.


### Profilo degli attempt

![Profilo degli attempt](../../results/nvidia_complete_api_test_RESULTS/p_valid_given_k.png)

**Come leggerlo:** Una curva piatta suggerisce re-sampling; una curva crescente suggerisce correzione; una curva decrescente spesso indica che i problemi facili vengono risolti subito e restano quelli difficili.

**Interconnessione:** P(Valid | reached attempt k) iteration profile — discriminates Stochastic Searcher (flat line) from Efficient Corrector (rising curve).


## Grounding, eseguibilit? e failure anatomy

Figure da `../../results/nvidia_complete_api_test_RESULTS/`.


### Grounding del vocabolario

![Grounding del vocabolario](../../results/nvidia_complete_api_test_RESULTS/hallucination_heatmap.png)

**Come leggerlo:** Misura azioni inventate rispetto agli operatori PDDL. Basso ? meglio. Se Halluc ? alto, Exec, PAS e CoT diventano meno affidabili perch? il piano non usa nemmeno il vocabolario corretto.

**Interconnessione:** Mean hallucination rate by model and domain. Lower is better.


### Hallucination per modello e dominio

![Hallucination per modello e dominio](../../results/nvidia_complete_api_test_RESULTS/hallucination_by_model_domain.png)

**Come leggerlo:** Aiuta a separare errori generali da domini in cui i nomi delle azioni sono pi? difficili o meno familiari.

**Interconnessione:** Action hallucination rate by model and domain.


### Oggetti inventati

![Oggetti inventati](../../results/nvidia_complete_api_test_RESULTS/object_hallucination_by_model_domain.png)

**Come leggerlo:** Misura argomenti non presenti nel problema PDDL. ? possibile conoscere le azioni legali ma inventare oggetti o location, quindi va distinto dall?action hallucination.

**Interconnessione:** Object hallucination rate by model and domain.


### Quanto il piano viene eseguito

![Quanto il piano viene eseguito](../../results/nvidia_complete_api_test_RESULTS/executability_by_model_domain.png)

**Come leggerlo:** Box alti e stretti indicano piani che arrivano lontano in modo consistente. Box bassi indicano fallimento precoce; box larghi indicano forte variabilit? tra istanze.

**Interconnessione:** Executability ratio distribution by model and domain.


### Esecuzione contro lunghezza del piano

Ogni figura e specifica per modello; dentro la figura i subplot separano i domini/task e i punti sono colorati per difficulty tier.

#### nvidia gpt oss 120b

![Esecuzione contro lunghezza del piano - nvidia gpt oss 120b](../../results/final_evaluation_results/executability_vs_length_nvidia_gpt_oss_120b.png)

#### nvidia kimi k2 6

![Esecuzione contro lunghezza del piano - nvidia kimi k2 6](../../results/final_evaluation_results/executability_vs_length_nvidia_kimi_k2_6.png)

#### nvidia llama 3 3 70b instruct

![Esecuzione contro lunghezza del piano - nvidia llama 3 3 70b instruct](../../results/final_evaluation_results/executability_vs_length_nvidia_llama_3_3_70b_instruct.png)

#### nvidia nemotron 3 nano 30b a3b

![Esecuzione contro lunghezza del piano - nvidia nemotron 3 nano 30b a3b](../../results/final_evaluation_results/executability_vs_length_nvidia_nemotron_3_nano_30b_a3b.png)

#### nvidia nemotron 3 ultra 550b a55b

![Esecuzione contro lunghezza del piano - nvidia nemotron 3 ultra 550b a55b](../../results/final_evaluation_results/executability_vs_length_nvidia_nemotron_3_ultra_550b_a55b.png)

#### nvidia phi 4 mini instruct

![Esecuzione contro lunghezza del piano - nvidia phi 4 mini instruct](../../results/final_evaluation_results/executability_vs_length_nvidia_phi_4_mini_instruct.png)

#### nvidia qwen 3 5 122b a10b

![Esecuzione contro lunghezza del piano - nvidia qwen 3 5 122b a10b](../../results/final_evaluation_results/executability_vs_length_nvidia_qwen_3_5_122b_a10b.png)

**Come leggerlo:** Se Exec cala con la lunghezza dentro molti domini/task, il modello perde coerenza di stato quando la catena causale diventa lunga. Se il calo appare solo in un subplot, il problema e piu probabilmente domain/task-specific.

**Interconnessione:** Executability ratio against plan length, one figure per model, faceted by domain/task and coloured by difficulty tier.


### Severit? degli errori di ordine

![Severit? degli errori di ordine](../../results/nvidia_complete_api_test_RESULTS/temporal_distance_by_model.png)

**Come leggerlo:** Distanze piccole sono near-miss; code lunghe indicano collasso del tracking temporale. Se il pannello manca o ? vuoto, potrebbero non esserci sequencing error classificabili.

**Interconnessione:** Mean temporal distance for sequencing errors per model.


### Sequencing vs state fabrication

![Sequencing vs state fabrication](../../results/nvidia_complete_api_test_RESULTS/failure_type_breakdown.png)

**Come leggerlo:** Sequencing error = il fatto richiesto era vero prima ma l?azione ? stata messa nel momento sbagliato. State fabrication = il fatto non ? mai stato prodotto; ? un errore pi? grave di world model.

**Interconnessione:** Sequencing errors versus state fabrications per model.


### Anatomia del fallimento

![Anatomia del fallimento](../../results/nvidia_complete_api_test_RESULTS/failure_mode_taxonomy.png)

**Come leggerlo:** Incrocia Halluc e PAS. Il quadrante migliore ha bassa hallucination e alto PAS; il peggiore ha alta hallucination e basso PAS.

**Interconnessione:** Failure taxonomy by hallucination rate and PAS.


## Chain-of-thought e faithful reasoning

Figure da `../../results/nvidia_complete_api_test_RESULTS/`.


### Faithfulness della reasoning trace

![Faithfulness della reasoning trace](../../results/nvidia_complete_api_test_RESULTS/cot_alignment_by_model_domain.png)

**Come leggerlo:** Mostra se il CoT parla delle stesse azioni e oggetti usati dal piano. Alto non basta: va controllato anche se il piano ? valido.

**Interconnessione:** Mean CoT plan alignment score by model and domain.


### CoT attivo contro non attivo

![CoT attivo contro non attivo](../../results/nvidia_complete_api_test_RESULTS/cot_success_rate.png)

**Come leggerlo:** Se CoT=True ha SR maggiore, controlla l?allineamento: un miglioramento con CoT basso pu? essere effetto del prompt, non reasoning fedele.

**Interconnessione:** Success rate split by CoT flag.


### Allineamento per piani validi/invalidi

![Allineamento per piani validi/invalidi](../../results/nvidia_complete_api_test_RESULTS/cot_alignment_validity.png)

**Come leggerlo:** Se i piani validi hanno allineamento pi? alto, il reasoning sembra contribuire. Se validi e invalidi sono simili, il CoT potrebbe essere decorativo.

**Interconnessione:** CoT plan alignment distribution for valid versus invalid plans.


## Confronto cross-domain e score composito

Figure da `../../results/nvidia_complete_api_test_RESULTS/`.


### Ranking normalizzato per dominio

Ogni figura e specifica per dominio. Le celle non sono valori metrici raw: sono rank normalizzati dentro quel dominio, con 1 = migliore e 0 = peggiore.

#### block grouping

![Ranking normalizzato per dominio - block grouping](../../results/final_evaluation_results/domain_ranking_heatmap_block_grouping.png)

#### expedition

![Ranking normalizzato per dominio - expedition](../../results/final_evaluation_results/domain_ranking_heatmap_expedition.png)

#### fo counters

![Ranking normalizzato per dominio - fo counters](../../results/final_evaluation_results/domain_ranking_heatmap_fo_counters.png)

#### fo sailing

![Ranking normalizzato per dominio - fo sailing](../../results/final_evaluation_results/domain_ranking_heatmap_fo_sailing.png)

#### rover

![Ranking normalizzato per dominio - rover](../../results/final_evaluation_results/domain_ranking_heatmap_rover.png)

#### settlersnumeric

![Ranking normalizzato per dominio - settlersnumeric](../../results/final_evaluation_results/domain_ranking_heatmap_settlersnumeric.png)

**Come leggerlo:** Confronta i modelli lungo una colonna dentro lo stesso dominio. Verde indica posizione relativa migliore per quella metrica; non significa necessariamente valore assoluto alto.

**Interconnessione:** Normalised within-domain rank heatmap, one plot per domain, with cells showing rank-derived values rather than raw scores.


### Generalista o specialista

![Generalista o specialista](../../results/nvidia_complete_api_test_RESULTS/rank_variance.png)

**Come leggerlo:** Varianza bassa = comportamento stabile tra domini; varianza alta = forte specializzazione o sensibilit? al dominio.

**Interconnessione:** Mean rank standard deviation across domains per model — high variance signals domain specialisation.


### Ridondanza tra domini

![Ridondanza tra domini](../../results/nvidia_complete_api_test_RESULTS/domain_correlation.png)

**Come leggerlo:** Correlazioni alte indicano domini che ordinano i modelli in modo simile. Correlazioni basse o negative indicano domini complementari e pi? informativi.

**Interconnessione:** Spearman correlation matrix of model success rates across domains — detects redundancy or complementarity.


### Score composito complessivo

![Score composito complessivo](../../results/nvidia_complete_api_test_RESULTS/composite_scores.png)

**Come leggerlo:** Riassume molte dimensioni, ma pu? nascondere trade-off. Usa il PS per ordinare, poi torna alle componenti per capire perch?.

**Interconnessione:** Composite Planning Score by model.


### Contributi del PS per dominio

![Contributi del PS per dominio](../../results/nvidia_complete_api_test_RESULTS/ps_by_domain_stacked.png)

**Come leggerlo:** Distingue modelli forti ovunque da modelli trainati da pochi domini. Una barra alta concentrata in pochi colori segnala specializzazione.

**Interconnessione:** Composite Planning Score stacked by domain contribution — reveals whether high PS is broad or concentrated.


### Tabella dei valori aggregati

![Tabella dei valori aggregati](../../results/nvidia_complete_api_test_RESULTS/metrics_summary_table.png)

**Come leggerlo:** ? la vista numerica da usare quando differenze visive piccole nel grafico sono difficili da stimare.

**Interconnessione:** Tabular summary of all key aggregate metrics rendered as a matplotlib figure for archiving.


### Profilo multidimensionale

![Profilo multidimensionale](../../results/nvidia_complete_api_test_RESULTS/radar_chart.png)

**Come leggerlo:** Area pi? ampia = profilo pi? forte, ma il radar enfatizza la forma. Usalo per vedere squilibri, non per fare ranking fine.

**Interconnessione:** Polar radar chart of FASR, IWSR, Exec, IHR, and PAS — larger filled area means stronger overall planning capability.


## Tabella sintetica ? LLM classification test

Set separato dal run NVIDIA completo. Non confrontare direttamente i valori assoluti se modelli, domini, protocolli o campionamento non coincidono.

| Model | N | SR | FASR | IWSR | Retry Gap | Exec | Halluc | IHR | PAS | CoT | PS |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| hf_gemma_4_31b_it | 71 | 0.042 | 0.014 | 0.026 | 0.028 | 0.482 | 0.000 | 1.000 | 0.419 | 0.000 | 0.350 |
| hf_nemotron_3_nano_30b_a3b | 52 | 0.000 | 0.000 | 0.000 | 0.000 | 0.604 | 0.000 | 1.000 | 0.104 | 0.000 | 0.320 |
| hf_phi_4 | 72 | 0.056 | 0.000 | 0.015 | 0.056 | 0.665 | 0.120 | 0.880 | 0.031 | 0.240 | 0.313 |
| hf_qwen_3_6_27b | 72 | 0.153 | 0.028 | 0.059 | 0.125 | 0.741 | 0.143 | 0.857 | 0.208 | 0.010 | 0.353 |


## Figure compatte ? LLM classification test

Queste sei figure sono una versione ridotta del report. Sono utili per una lettura rapida, ma coprono meno diagnostica rispetto alle 27 figure del run completo.


### Success Rate Heatmap

![Success Rate Heatmap](../../results/LLM_classification_test/01_success_rate_heatmap.png)

**Come leggerlo:** Leggi le celle come quota di piani validi per modello e dominio. ? una vista compatta dell?outcome, quindi va sempre controllata con FASR/IWSR se disponibili.


### Hallucination Heatmap

![Hallucination Heatmap](../../results/LLM_classification_test/02_hallucination_heatmap.png)

**Come leggerlo:** Basso ? meglio. Se un modello inventa molte azioni, le metriche di esecuzione successive diventano meno interpretabili.


### Composite Score Bar

![Composite Score Bar](../../results/LLM_classification_test/03_composite_score_bar.png)

**Come leggerlo:** Classifica sintetica. ? utile per ordinare i modelli, ma bisogna verificare quali componenti hanno prodotto il punteggio.


### Radar Chart

![Radar Chart](../../results/LLM_classification_test/04_radar_chart.png)

**Come leggerlo:** Mostra il profilo delle capacit?: non guardare solo l?area, ma anche quali assi sono deboli.


### Failure Taxonomy

![Failure Taxonomy](../../results/LLM_classification_test/05_failure_taxonomy.png)

**Come leggerlo:** Posiziona i modelli rispetto al tipo di errore. Serve a distinguere fallimenti di grounding da fallimenti di sequencing.


### FASR vs IWSR

![FASR vs IWSR](../../results/LLM_classification_test/06_fasr_vs_iwsr.png)

**Come leggerlo:** Se un punto ? molto sopra la diagonale, il modello dipende dai retry; se ? vicino alla diagonale con valori alti, ? pi? forte al primo tentativo.


## Appendice ? prova_merge

Queste immagini sono incluse perch? presenti in `results/prova_merge/`. Non ho trovato un JSON di accompagnamento nella cartella `results/`, quindi qui non aggiungo tabelle numeriche n? conclusioni specifiche. Usale come esportazione separata: confrontare direttamente `prova_merge` con `nvidia_complete_api_test_RESULTS` ? corretto solo se sai che run, protocolli, domini e criteri di merge coincidono.


### prova_merge ? Outcome per dominio

![prova_merge ? Outcome per dominio](../../results/prova_merge/success_rate_heatmap.png)

**Lettura:** Ogni cella mostra la quota di problemi validi per una coppia modello-dominio. Serve per capire dove un modello funziona, ma non dice se il successo arriva al primo tentativo o dopo molti retry.


### prova_merge ? Successo totale contro primo tentativo

![prova_merge ? Successo totale contro primo tentativo](../../results/prova_merge/sr_vs_fasr.png)

**Lettura:** La distanza tra SR e FASR segnala dipendenza dal meccanismo iterativo. Se le barre sono vicine, il modello risolve soprattutto subito; se SR ? molto pi? alto, il risultato finale ? scaffolded dai retry.


### prova_merge ? SR, FASR e IWSR insieme

![prova_merge ? SR, FASR e IWSR insieme](../../results/prova_merge/sr_fasr_iwsr_by_model.png)

**Lettura:** Usa SR come tetto finale, FASR come baseline zero-shot e IWSR come efficienza dei retry. IWSR vicino a SR indica correzioni rapide; IWSR vicino a FASR indica che i retry aggiungono poco o arrivano tardi.


### prova_merge ? FASR per dominio

![prova_merge ? FASR per dominio](../../results/prova_merge/fasr_by_model_domain.png)

**Lettura:** Mostra la robustezza al primo tentativo. ? la vista pi? severa perch? non pu? essere gonfiata dal numero di iterazioni.


### prova_merge ? FASR per difficolt?

![prova_merge ? FASR per difficolt?](../../results/prova_merge/fasr_by_difficulty.png)

**Lettura:** Controlla se la capacit? zero-shot degrada da easy a hard. Un crollo netto indica sensibilit? alla complessit?; una linea stabile indica maggiore generalizzazione.


### prova_merge ? Efficienza delle correzioni

![prova_merge ? Efficienza delle correzioni](../../results/prova_merge/iwsr_by_model_domain.png)

**Lettura:** Confronta quanto i retry sono utili nei diversi domini. Va letto con FASR: alto IWSR con basso FASR pu? indicare buon correttore, non necessariamente buon planner zero-shot.


### prova_merge ? Dipendenza dai retry

![prova_merge ? Dipendenza dai retry](../../results/prova_merge/retry_gap_by_model.png)

**Lettura:** Valori alti significano che molti successi spariscono togliendo il meccanismo iterativo. Non interpretare Retry Gap basso come qualit? se anche SR ? vicino a zero.


### prova_merge ? Profilo zero-shot / retry

![prova_merge ? Profilo zero-shot / retry](../../results/prova_merge/fasr_iwsr_scatter.png)

**Lettura:** Asse X = FASR, asse Y = IWSR, dimensione/colore = SR. Punti in alto a destra sono migliori; punti molto sopra la diagonale dipendono dai retry.


### prova_merge ? Profilo degli attempt

![prova_merge ? Profilo degli attempt](../../results/prova_merge/p_valid_given_k.png)

**Lettura:** Una curva piatta suggerisce re-sampling; una curva crescente suggerisce correzione; una curva decrescente spesso indica che i problemi facili vengono risolti subito e restano quelli difficili.


### prova_merge ? Grounding del vocabolario

![prova_merge ? Grounding del vocabolario](../../results/prova_merge/hallucination_heatmap.png)

**Lettura:** Misura azioni inventate rispetto agli operatori PDDL. Basso ? meglio. Se Halluc ? alto, Exec, PAS e CoT diventano meno affidabili perch? il piano non usa nemmeno il vocabolario corretto.


### prova_merge ? Hallucination per modello e dominio

![prova_merge ? Hallucination per modello e dominio](../../results/prova_merge/hallucination_by_model_domain.png)

**Lettura:** Aiuta a separare errori generali da domini in cui i nomi delle azioni sono pi? difficili o meno familiari.


### prova_merge ? Oggetti inventati

![prova_merge ? Oggetti inventati](../../results/prova_merge/object_hallucination_by_model_domain.png)

**Lettura:** Misura argomenti non presenti nel problema PDDL. ? possibile conoscere le azioni legali ma inventare oggetti o location, quindi va distinto dall?action hallucination.


### prova_merge ? Quanto il piano viene eseguito

![prova_merge ? Quanto il piano viene eseguito](../../results/prova_merge/executability_by_model_domain.png)

**Lettura:** Box alti e stretti indicano piani che arrivano lontano in modo consistente. Box bassi indicano fallimento precoce; box larghi indicano forte variabilit? tra istanze.


### Esecuzione contro lunghezza del piano

Ogni figura e specifica per modello; dentro la figura i subplot separano i domini/task e i punti sono colorati per difficulty tier.

#### nvidia gpt oss 120b

![Esecuzione contro lunghezza del piano - nvidia gpt oss 120b](../../results/final_evaluation_results/executability_vs_length_nvidia_gpt_oss_120b.png)

#### nvidia kimi k2 6

![Esecuzione contro lunghezza del piano - nvidia kimi k2 6](../../results/final_evaluation_results/executability_vs_length_nvidia_kimi_k2_6.png)

#### nvidia llama 3 3 70b instruct

![Esecuzione contro lunghezza del piano - nvidia llama 3 3 70b instruct](../../results/final_evaluation_results/executability_vs_length_nvidia_llama_3_3_70b_instruct.png)

#### nvidia nemotron 3 nano 30b a3b

![Esecuzione contro lunghezza del piano - nvidia nemotron 3 nano 30b a3b](../../results/final_evaluation_results/executability_vs_length_nvidia_nemotron_3_nano_30b_a3b.png)

#### nvidia nemotron 3 ultra 550b a55b

![Esecuzione contro lunghezza del piano - nvidia nemotron 3 ultra 550b a55b](../../results/final_evaluation_results/executability_vs_length_nvidia_nemotron_3_ultra_550b_a55b.png)

#### nvidia phi 4 mini instruct

![Esecuzione contro lunghezza del piano - nvidia phi 4 mini instruct](../../results/final_evaluation_results/executability_vs_length_nvidia_phi_4_mini_instruct.png)

#### nvidia qwen 3 5 122b a10b

![Esecuzione contro lunghezza del piano - nvidia qwen 3 5 122b a10b](../../results/final_evaluation_results/executability_vs_length_nvidia_qwen_3_5_122b_a10b.png)

**Come leggerlo:** Se Exec cala con la lunghezza dentro molti domini/task, il modello perde coerenza di stato quando la catena causale diventa lunga. Se il calo appare solo in un subplot, il problema e piu probabilmente domain/task-specific.

**Interconnessione:** Executability ratio against plan length, one figure per model, faceted by domain/task and coloured by difficulty tier.


### prova_merge ? Severit? degli errori di ordine

![prova_merge ? Severit? degli errori di ordine](../../results/prova_merge/temporal_distance_by_model.png)

**Lettura:** Distanze piccole sono near-miss; code lunghe indicano collasso del tracking temporale. Se il pannello manca o ? vuoto, potrebbero non esserci sequencing error classificabili.


### prova_merge ? Sequencing vs state fabrication

![prova_merge ? Sequencing vs state fabrication](../../results/prova_merge/failure_type_breakdown.png)

**Lettura:** Sequencing error = il fatto richiesto era vero prima ma l?azione ? stata messa nel momento sbagliato. State fabrication = il fatto non ? mai stato prodotto; ? un errore pi? grave di world model.


### prova_merge ? Anatomia del fallimento

![prova_merge ? Anatomia del fallimento](../../results/prova_merge/failure_mode_taxonomy.png)

**Lettura:** Incrocia Halluc e PAS. Il quadrante migliore ha bassa hallucination e alto PAS; il peggiore ha alta hallucination e basso PAS.


### prova_merge ? Faithfulness della reasoning trace

![prova_merge ? Faithfulness della reasoning trace](../../results/prova_merge/cot_alignment_by_model_domain.png)

**Lettura:** Mostra se il CoT parla delle stesse azioni e oggetti usati dal piano. Alto non basta: va controllato anche se il piano ? valido.


### prova_merge ? CoT attivo contro non attivo

![prova_merge ? CoT attivo contro non attivo](../../results/prova_merge/cot_success_rate.png)

**Lettura:** Se CoT=True ha SR maggiore, controlla l?allineamento: un miglioramento con CoT basso pu? essere effetto del prompt, non reasoning fedele.


### prova_merge ? Allineamento per piani validi/invalidi

![prova_merge ? Allineamento per piani validi/invalidi](../../results/prova_merge/cot_alignment_validity.png)

**Lettura:** Se i piani validi hanno allineamento pi? alto, il reasoning sembra contribuire. Se validi e invalidi sono simili, il CoT potrebbe essere decorativo.


### Ranking normalizzato per dominio

Ogni figura e specifica per dominio. Le celle non sono valori metrici raw: sono rank normalizzati dentro quel dominio, con 1 = migliore e 0 = peggiore.

#### block grouping

![Ranking normalizzato per dominio - block grouping](../../results/final_evaluation_results/domain_ranking_heatmap_block_grouping.png)

#### expedition

![Ranking normalizzato per dominio - expedition](../../results/final_evaluation_results/domain_ranking_heatmap_expedition.png)

#### fo counters

![Ranking normalizzato per dominio - fo counters](../../results/final_evaluation_results/domain_ranking_heatmap_fo_counters.png)

#### fo sailing

![Ranking normalizzato per dominio - fo sailing](../../results/final_evaluation_results/domain_ranking_heatmap_fo_sailing.png)

#### rover

![Ranking normalizzato per dominio - rover](../../results/final_evaluation_results/domain_ranking_heatmap_rover.png)

#### settlersnumeric

![Ranking normalizzato per dominio - settlersnumeric](../../results/final_evaluation_results/domain_ranking_heatmap_settlersnumeric.png)

**Come leggerlo:** Confronta i modelli lungo una colonna dentro lo stesso dominio. Verde indica posizione relativa migliore per quella metrica; non significa necessariamente valore assoluto alto.

**Interconnessione:** Normalised within-domain rank heatmap, one plot per domain, with cells showing rank-derived values rather than raw scores.


### prova_merge ? Generalista o specialista

![prova_merge ? Generalista o specialista](../../results/prova_merge/rank_variance.png)

**Lettura:** Varianza bassa = comportamento stabile tra domini; varianza alta = forte specializzazione o sensibilit? al dominio.


### prova_merge ? Ridondanza tra domini

![prova_merge ? Ridondanza tra domini](../../results/prova_merge/domain_correlation.png)

**Lettura:** Correlazioni alte indicano domini che ordinano i modelli in modo simile. Correlazioni basse o negative indicano domini complementari e pi? informativi.


### prova_merge ? Score composito complessivo

![prova_merge ? Score composito complessivo](../../results/prova_merge/composite_scores.png)

**Lettura:** Riassume molte dimensioni, ma pu? nascondere trade-off. Usa il PS per ordinare, poi torna alle componenti per capire perch?.


### prova_merge ? Contributi del PS per dominio

![prova_merge ? Contributi del PS per dominio](../../results/prova_merge/ps_by_domain_stacked.png)

**Lettura:** Distingue modelli forti ovunque da modelli trainati da pochi domini. Una barra alta concentrata in pochi colori segnala specializzazione.


### prova_merge ? Tabella dei valori aggregati

![prova_merge ? Tabella dei valori aggregati](../../results/prova_merge/metrics_summary_table.png)

**Lettura:** ? la vista numerica da usare quando differenze visive piccole nel grafico sono difficili da stimare.


### prova_merge ? Profilo multidimensionale

![prova_merge ? Profilo multidimensionale](../../results/prova_merge/radar_chart.png)

**Lettura:** Area pi? ampia = profilo pi? forte, ma il radar enfatizza la forma. Usalo per vedere squilibri, non per fare ranking fine.


## Problemi interpretativi da controllare

- **Path statici:** le immagini sono linkate da `results/`. Se il notebook viene spostato fuori da `analysis/notebooks/`, i link vanno aggiornati.
- **Snapshot, non pipeline:** questo file non rigenera dati. Se cambiano i JSON o i PNG, bisogna rigenerare il report statico.
- **Set non equivalenti:** `nvidia_complete_api_test_RESULTS`, `LLM_classification_test` e `prova_merge` non vanno fusi mentalmente senza verificare run, domini, protocolli e criteri di aggregazione.
- **Retry:** SR alto con FASR basso non ? prova di planning forte; pu? essere dipendenza dallo scaffold iterativo.
- **Hallucination gate:** se `Halluc` ? alto o `IHR` ? basso, Exec, PAS e CoT possono descrivere artefatti di parsing pi? che capacit? reale.
- **PAS e Temporal Distance:** valori mancanti possono indicare assenza di errori classificabili, piani completamente eseguibili o fallimenti non interpretabili; controlla sempre failure breakdown.
- **CoT:** alignment alto indica sovrapposizione tra reasoning trace e piano, non causalit? garantita. Serve confrontarlo con validit? e outcome.
- **Composite score:** PS ? utile per ordinare, ma non sostituisce l?analisi delle componenti; due modelli con PS simile possono fallire per ragioni opposte.
